# Data Pre-processing

In [36]:
# Set up project root
import sys
import os
current_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(current_dir, '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

## eBird Data Prep

In [37]:
from data_handler import eBirdDataHandler
ebird_data_handler = eBirdDataHandler()

In [38]:
### Checklist Data
checklists_df = await ebird_data_handler.get_checklists_data()
print(checklists_df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24346 entries, 0 to 24345
Data columns (total 9 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   isoObsDate       24346 non-null  object
 1   loc              24346 non-null  object
 2   locId            24346 non-null  object
 3   numSpecies       24346 non-null  int64 
 4   obsDt            24346 non-null  object
 5   obsTime          24279 non-null  object
 6   subID            24346 non-null  object
 7   subId            24346 non-null  object
 8   userDisplayName  24345 non-null  object
dtypes: int64(1), object(8)
memory usage: 1.7+ MB
None


In [39]:
### Checklist Record Data
checklist_records_df = await ebird_data_handler.get_checklist_records_data()
print(checklist_records_df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24346 entries, 0 to 24345
Data columns (total 25 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   allObsReported               24346 non-null  bool   
 1   checklistId                  24346 non-null  object 
 2   creationDt                   24346 non-null  object 
 3   deleteTrack                  24346 non-null  bool   
 4   durationHrs                  21408 non-null  float64
 5   effortDistanceEnteredUnit    10998 non-null  object 
 6   effortDistanceKm             10713 non-null  float64
 7   lastEditedDt                 24346 non-null  object 
 8   locId                        24346 non-null  object 
 9   numObservers                 24120 non-null  float64
 10  numSpecies                   24346 non-null  int64  
 11  obs                          24346 non-null  object 
 12  obsDt                        24346 non-null  object 
 13  obsTimeValid    

In [40]:
### Location Data
locations_df = await ebird_data_handler.get_location_data()
print(locations_df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7583 entries, 0 to 7582
Data columns (total 16 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   countryCode       7583 non-null   object 
 1   countryName       7583 non-null   object 
 2   hierarchicalName  7583 non-null   object 
 3   isHotspot         7583 non-null   bool   
 4   lat               7583 non-null   float64
 5   latitude          7583 non-null   float64
 6   lng               7583 non-null   float64
 7   locID             7583 non-null   object 
 8   locId             7583 non-null   object 
 9   locName           7583 non-null   object 
 10  longitude         7583 non-null   float64
 11  name              7583 non-null   object 
 12  subnational1Code  7583 non-null   object 
 13  subnational1Name  7583 non-null   object 
 14  subnational2Code  7580 non-null   object 
 15  subnational2Name  7580 non-null   object 
dtypes: bool(1), float64(4), object(11)
memory 

## Weather Data

In [41]:
from data_handler import WeatherDataHandler

weather_data_handler = WeatherDataHandler()

weather_df = await weather_data_handler.get_weather_data()
print(weather_df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3777912 entries, 0 to 3777911
Data columns (total 14 columns):
 #   Column                Dtype  
---  ------                -----  
 0   date                  object 
 1   temperature_2m        float64
 2   apparent_temperature  float64
 3   rain                  float64
 4   weather_code          float64
 5   cloud_cover           float64
 6   cloud_cover_mid       float64
 7   cloud_cover_high      float64
 8   cloud_cover_low       float64
 9   wind_speed_10m        float64
 10  wind_speed_100m       float64
 11  wind_direction_100m   float64
 12  wind_direction_10m    float64
 13  locId                 object 
dtypes: float64(12), object(2)
memory usage: 403.5+ MB
None


## Combined Data Set

In [42]:
import pandas as pd

# Combine Checklists and Checklist Records
checklists_expanded_records = pd.merge(checklists_df, checklist_records_df, on=['subId','locId', 'numSpecies', 'userDisplayName'], how='left')
checklists_expanded_locations = pd.merge(checklists_expanded_records, locations_df, on=['locId', 'subnational1Code'], how='left')
print(checklists_expanded_locations.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 39895 entries, 0 to 39894
Data columns (total 44 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   isoObsDate                   39895 non-null  object 
 1   loc                          39895 non-null  object 
 2   locId                        39895 non-null  object 
 3   numSpecies                   39895 non-null  int64  
 4   obsDt_x                      39895 non-null  object 
 5   obsTime                      39800 non-null  object 
 6   subID                        39895 non-null  object 
 7   subId                        39895 non-null  object 
 8   userDisplayName              39893 non-null  object 
 9   allObsReported               39517 non-null  object 
 10  checklistId                  39517 non-null  object 
 11  creationDt                   39517 non-null  object 
 12  deleteTrack                  39517 non-null  object 
 13  durationHrs     

In [43]:
# Combine Weather data

# Prepare super table
checklists_expanded_locations['isoObsDate'] = pd.to_datetime(checklists_expanded_locations['isoObsDate'])
# Convert the Indian Standard Time(IST) to UTC
checklists_expanded_locations['isoObsDate'] = checklists_expanded_locations['isoObsDate'].dt.tz_localize('Asia/Kolkata').dt.tz_convert('UTC')
checklists_expanded_locations = checklists_expanded_locations.rename(columns={'isoObsDate': 'merge_date'})

# Prepare weather table
weather_df['date'] = pd.to_datetime(weather_df['date'])
weather_df = weather_df.rename(columns={'date': 'merge_date'})

# Sort data frames (Required step)
checklists_expanded_locations = checklists_expanded_locations.sort_values('merge_date')
weather_df = weather_df.sort_values('merge_date')

# Merge weather data for nearest time in checklist table
df = pd.merge_asof(
    checklists_expanded_locations, 
    weather_df, 
    on='merge_date', 
    direction='nearest',
    tolerance=pd.Timedelta('1 hour'), # Only match if weather time is within 1 hour of checklist data
    suffixes=('', '_from_weather') # In case of overlapping column names
)

print(df.info())



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 39895 entries, 0 to 39894
Data columns (total 57 columns):
 #   Column                       Non-Null Count  Dtype              
---  ------                       --------------  -----              
 0   merge_date                   39895 non-null  datetime64[ns, UTC]
 1   loc                          39895 non-null  object             
 2   locId                        39895 non-null  object             
 3   numSpecies                   39895 non-null  int64              
 4   obsDt_x                      39895 non-null  object             
 5   obsTime                      39800 non-null  object             
 6   subID                        39895 non-null  object             
 7   subId                        39895 non-null  object             
 8   userDisplayName              39893 non-null  object             
 9   allObsReported               39517 non-null  object             
 10  checklistId                  39517 non-null  o